# Cartographie Sémantique et Analyse des Publications de la FSBM par NLP & Web Scraping

**Faculté des Sciences Ben M'Sik (FSBM) — Université Hassan II de Casablanca**

Ce notebook raconte le pipeline de bout en bout : liste des chercheurs (PDF) → profils Google Scholar → publications →
abstracts → nettoyage → embeddings **zembed-1** → index FAISS → recherche sémantique → cartographie.

> **Règle du projet :** aucune donnée scientifique n'est inventée. Toute valeur introuvable reste `None`/`NaN`
> avec un statut qui explique pourquoi. Ce notebook **lit** les résultats produits par les scripts `scripts/01…08` ;
> chaque section signale clairement si une étape préalable n'a pas encore été exécutée.

## 1. Introduction
L'objectif est d'évaluer et de valoriser la production scientifique des enseignants-chercheurs de la FSBM en
construisant un jeu de données propre, exploitable par des techniques de NLP.

## 2. Objectif
1. Identifier les chercheurs FSBM (source de référence : `Membres FSBM.pdf`) ;
2. retrouver **prudemment** leurs profils Google Scholar ;
3. collecter publications, métriques et abstracts (Scholar puis Crossref / OpenAlex / Semantic Scholar) ;
4. nettoyer, dédupliquer, valider ;
5. encoder titre + abstract avec **zembed-1** ;
6. offrir une recherche sémantique de publications et de chercheurs ;
7. produire une cartographie sémantique et des analyses descriptives.

## 3. Architecture du pipeline
```mermaid
graph TD
A[PDF Membres FSBM] --> B[Extraction chercheurs]
B --> C[Google Scholar]
C --> D[Publications]
D --> E[Enrichissement abstracts]
E --> F[Data Cleaning]
F --> G[zembed-1]
G --> H[FAISS]
H --> I[Recherche sémantique]
G --> J[Cartographie sémantique]
```

| Étape | Script | Sortie principale |
|---|---|---|
| 0 | `01_extract_members.py` | `data/raw/chercheurs_fsbm.csv` |
| 1 | `02_scrape_scholar.py` | `data/raw/scholars_raw.json`, `publications_raw.json` |
| 2 | `03_clean_data.py` | `data/processed/*` + rapport qualité |
| 3 | `04_generate_embeddings.py` | `data/vector_store/embeddings.npy` |
| 3 | `05_build_index.py` | index FAISS |
| 3 | `06_demo_search.py` | résultats de recherche |
| + | `07_semantic_map.py`, `08_descriptive_analysis.py` | figures |

In [ ]:
import os, sys, json, warnings
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "config" / "settings.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display

from src.utils.config import load_settings, resolve_path, get_env
from src.utils.io import read_json

pd.set_option("display.max_colwidth", 80, "display.width", 160)
settings = load_settings(os.environ.get("FSBM_SETTINGS"))   # FSBM_SETTINGS : config alternative (optionnel)
RAW, PROC = resolve_path(settings, "raw_dir"), resolve_path(settings, "processed_dir")
STORE, FIG, REP = (resolve_path(settings, k) for k in ("vector_store_dir", "figures_dir", "reports_dir"))


def available(*paths, hint="") -> bool:
    "Vrai si tous les fichiers existent ; sinon explique quelle étape lancer."
    missing = [Path(p).name for p in paths if not Path(p).exists()]
    if missing:
        print(f"⚠ Fichier(s) absent(s) : {missing}. {hint}")
    return not missing


print("Racine du projet :", ROOT)

## 4. Chargement de la liste FSBM
Le PDF `Membres FSBM.pdf` est la **source de référence** (colonnes : Etablissement, Enseignant Chercheur, Laboratoire, Equipe,
Type Membre). `scripts/01_extract_members.py` l'extrait avec `pdfplumber` par coordonnées de mots (impression d’une page web, contrôle d’intégrité 1…N et total « Records »), avec repli (tableaux →
repli texte), normalise les noms (`DRISS.BOUGGAR` → `Driss Bouggar`, valeur originale conservée) et crée un identifiant stable
(`fsbm_driss_bouggar`). Les membres d'autres établissements (ENCG, FSJESAS…) restent dans le fichier brut.

In [ ]:
members = None
if available(RAW / "chercheurs_fsbm.csv", hint="Lancez : python scripts/01_extract_members.py"):
    members = pd.read_csv(RAW / "chercheurs_fsbm.csv")
    display(members.head(10))
    print(members.shape)

## 5. Exploration des chercheurs

In [ ]:
if members is not None:
    from src.data.extract_fsbm_members import summarize_members
    stats = summarize_members(members, settings["data"]["institution"])
    for key in ("lignes_extraites", "personnes_uniques_total", "membres_fsbm_uniques", "etablissements",
                "laboratoires_fsbm", "equipes_fsbm", "lignes_doublons_exacts", "personnes_avec_plusieurs_affectations",
                "valeurs_manquantes"):
        print(f"{key:40s}: {stats[key]}")
    display(pd.Series(stats["chercheurs_par_laboratoire"], name="chercheurs").to_frame())

## 6. Scraping Google Scholar
`scripts/02_scrape_scholar.py` collecte, pour chaque chercheur FSBM :
* **conformité `robots.txt`** : Google Scholar autorise les profils (`/citations?user=ID`) mais interdit la recherche d'auteurs par nom et la
  pagination (`cstart=`). Par défaut, chaque URL est vérifiée avant envoi et les profils sont identifiés par un **Scholar ID renseigné à la main**
  (`--init-overrides` génère le modèle `data/raw/scholar_overrides.csv`) ; `--allow-author-search` réactive la recherche par nom, sous la
  responsabilité de l'utilisateur ;
* **résolution prudente du profil** (recherche par nom, si activée) : score de confiance (`profile_match_confidence`) combinant nom, affiliation
  (« Faculty of science Ben M'Sik », « Hassan II »), domaine d'email `univh2c.ma`, laboratoire et co-auteurs FSBM. Le nom seul plafonne à 0,40.
  Statuts : `matched`, `ambiguous`, `not_found`, `blocked`, `error`. Les cas ambigus sont exportés pour **vérification manuelle**
  (`data/raw/scholar_candidates_review.csv`) ;
* **robustesse** : délai aléatoire, backoff exponentiel, retries limités, cache par chercheur, checkpoint après chaque chercheur,
  reprise (`--resume`). En cas de CAPTCHA/429 : **arrêt propre**, aucune tentative de contournement ;
* **enrichissement** des abstracts : Scholar → Crossref → OpenAlex → Semantic Scholar (garde-fou sur la similarité du titre).

```bash
python scripts/02_scrape_scholar.py --init-overrides                              # modèle des Scholar ID
python scripts/02_scrape_scholar.py --limit-researchers 3 --max-publications 10   # test
python scripts/02_scrape_scholar.py --resume                                      # exécution complète / reprise
```

In [ ]:
scholars = pubs_raw = None
if available(RAW / "scholars_raw.json", RAW / "publications_raw.json", hint="Lancez : python scripts/02_scrape_scholar.py"):
    scholars = pd.DataFrame(read_json(RAW / "scholars_raw.json"))
    pubs_raw = pd.DataFrame(read_json(RAW / "publications_raw.json"))
    print("Statuts de profil Scholar :")
    display(scholars["scholar_profile_status"].value_counts().to_frame())
    print("Statut de collecte :")
    display(scholars["collection_status"].value_counts().to_frame())
    print("Confiance d'appariement (profils matched) :")
    display(scholars.loc[scholars.scholar_profile_status == "matched", "profile_match_confidence"].describe().to_frame().T)
    amb = scholars[scholars.scholar_profile_status == "ambiguous"]
    if len(amb):
        print(f"{len(amb)} profil(s) ambigu(s) à vérifier manuellement (data/raw/scholar_candidates_review.csv)")
    print("Publications brutes :", pubs_raw.shape)
    if len(pubs_raw):
        display(pubs_raw[["chercheur_id", "titre", "annee", "citations", "abstract_status", "scrape_status"]].head())

## 7. Qualité des données
Le rapport est produit par `scripts/03_clean_data.py` (`outputs/reports/data_quality_report.json`). Les pourcentages décrivent
**le dataset collecté** — pas la production totale de la FSBM.

In [ ]:
if available(REP / "data_quality_report.json", hint="Lancez : python scripts/03_clean_data.py"):
    report = read_json(REP / "data_quality_report.json")
    for k, v in report.items():
        print(f"{k:34s}: {v}")

## 8. Nettoyage
* texte : Unicode NFC, suppression HTML/JATS, caractères de contrôle, espaces multiples ; **symboles scientifiques conservés** (α, µ, ±, °) ;
* deux versions de l'abstract : `abstract` (brut, jamais modifié) et `abstract_clean` (minuscules, sans étiquette « Abstract », sans « … » de troncature) ;
* **pas de suppression de stop-words ni de stemming** : les modèles d'embeddings modernes exploitent la phrase entière ;
* dates → ISO à la précision connue (`date_precision` : day/month/year) ; métriques négatives, années implausibles, URL invalides → `None` ;
* déduplication : DOI normalisé → titre normalisé + année → flou prudent (jamais deux DOI différents, jamais « Part I » / « Part II »).

In [ ]:
from src.preprocessing.text_cleaner import clean_abstract_for_nlp
from src.preprocessing.data_cleaner import parse_publication_date

raw_example = "<jats:p>Abstract: We STUDY   CO2 capture at 25 °C   in Zeolites…</jats:p>"
print("brut :", raw_example)
print("clean:", clean_abstract_for_nlp(raw_example))
for d in ["2021/1/1", "2021/3", "2019", "March 5, 2018", None]:
    print(f"{d!s:15} ->", parse_publication_date(d))

In [ ]:
articles = researchers = links = None
if available(PROC / "publications_clean.parquet", PROC / "chercheurs_clean.csv", PROC / "researcher_publications.csv",
             hint="Lancez : python scripts/03_clean_data.py"):
    articles = pd.read_parquet(PROC / "publications_clean.parquet")
    researchers = pd.read_csv(PROC / "chercheurs_clean.csv")
    links = pd.read_csv(PROC / "researcher_publications.csv")
    print(f"{len(articles)} publications uniques | {len(links)} liens chercheur↔publication | {len(researchers)} chercheurs")
    if len(articles):
        print("\nPart de valeurs manquantes (%) :")
        display((articles.isna().mean() * 100).round(1).sort_values(ascending=False).head(10).to_frame("% manquant"))
        row = articles[articles.abstract.notna()].head(1)
        if len(row):
            r = row.iloc[0]
            print("\nabstract brut  :", str(r.abstract)[:200])
            print("abstract_clean :", str(r.abstract_clean)[:200])

## 9. Analyse exploratoire

In [ ]:
if articles is not None and len(articles):
    from src.analysis.descriptive_analysis import run_descriptive_analysis
    members_all = pd.read_csv(RAW / "chercheurs_fsbm.csv")
    members_all = members_all[members_all["chercheur_id"].isin(researchers["chercheur_id"])]   # mêmes chercheurs que le dataset
    result = run_descriptive_analysis(members_all, researchers, articles, links, FIG, REP, settings["data"]["institution"])
    for path in result["figures"]:
        display(Image(filename=path, width=700))
    print("⚠ Ces classements décrivent le dataset collecté, pas la production totale de la FSBM.")

## 10. Préparation NLP
Texte encodé = **titre + abstract nettoyé**. Sans abstract exploitable (absent, ou plus court que `min_abstract_chars`), on encode le titre seul et on
l'indique dans `embedding_source` (`title_abstract` | `title_only`).

In [ ]:
if articles is not None and len(articles):
    display(articles["embedding_source"].value_counts().to_frame())
    lengths = articles["embedding_text"].dropna().str.len()
    lengths.plot.hist(bins=30, figsize=(7, 3), title="Longueur du texte encodé (caractères)")
    plt.show()
    print(articles["embedding_text"].dropna().iloc[0][:300])

## 11. zembed-1
Modèle d'embedding **zembed-1** de **ZeroEntropy**. L'éditeur n'accepte plus de nouvelles inscriptions (plus de clé API possible) ;
le modèle est cependant publié en **poids ouverts** (`zeroentropy/zembed-1-embedding`, Hugging Face, Apache-2.0, 4 Md de paramètres,
base Qwen3-4B), exécutés avec `sentence-transformers`. C'est **le même modèle**, pas un substitut : le client refuse toute autre valeur.

* `encode_document` pour les publications, `encode_query` pour les requêtes ; dimension **2560** par défaut (Matryoshka : 1280…40), lue sur les vecteurs, jamais supposée ;
* ~8 Go de poids : l'encodage des publications se fait sur **GPU Colab** (`notebooks/02_colab_embeddings_gpu.ipynb`) ;
* les 5 requêtes de démonstration sont encodées en même temps et stockées dans le cache SQLite : la recherche ne recharge pas le modèle.

In [ ]:
run_info = read_json(STORE / "embedding_run.json")
if run_info:
    print("Dernière génération :", {k: run_info.get(k) for k in ("model", "transport", "runtime", "dimension", "n_embedded", "demo_queries_cached")})
client = None
if (STORE / "embedding_cache.sqlite").exists():
    from src.embeddings.zembed_client import ZEmbedClient
    # les poids (~8 Go) ne sont chargés que si un texte n'est pas déjà dans le cache
    client = ZEmbedClient.from_settings(settings, cache_path=STORE / "embedding_cache.sqlite")
    print("Client zembed-1 prêt — transport :", client.transport, "| entrées en cache :", len(client.cache))
else:
    print("Pas de cache d'embeddings : exécutez le notebook Colab (ou scripts/04) puis copiez les fichiers dans data/vector_store/.")

## 12. Vectorisation

In [ ]:
vectors = manifest = None
if available(STORE / "embeddings.npy", STORE / "embedding_manifest.csv", hint="Lancez : python scripts/04_generate_embeddings.py"):
    vectors = np.load(STORE / "embeddings.npy")
    manifest = pd.read_csv(STORE / "embedding_manifest.csv")
    print("embeddings :", vectors.shape, vectors.dtype, "| modèle du manifest :", manifest["model"].unique())
    norms = np.linalg.norm(vectors, axis=1)
    print("normes L2 : min %.3f, max %.3f" % (norms.min(), norms.max()))
    from src.search.vector_store import cosine_similarity
    sample = vectors[: min(200, len(vectors))]
    sims = cosine_similarity(sample, sample)
    iu = np.triu_indices_from(sims, k=1)
    plt.figure(figsize=(7, 3))
    plt.hist(sims[iu], bins=40)
    plt.title("Similarités cosinus entre publications (échantillon)")
    plt.show()

## 13. Index FAISS
Vecteurs **normalisés L2** + `IndexFlatIP` = similarité **cosinus exacte**. Sont sauvegardés : l'index, la correspondance `vector_id → article_id` et les métadonnées.

In [ ]:
store = None
if available(STORE / "index_info.json", hint="Lancez : python scripts/05_build_index.py"):
    from src.search.vector_store import VectorStore
    store = VectorStore.load(STORE)
    print(json.dumps(store.info, indent=2, ensure_ascii=False))
    if vectors is not None:   # contrôle de cohérence : un vecteur doit se retrouver lui-même avec un score ≈ 1
        hit = store.search(vectors[:1], 1)[0][0]
        print("auto-récupération :", hit[0], "score = %.4f" % hit[1], "(attendu ≈ 1.0000)")

## 14. Recherche sémantique
`semantic_search(query, top_k=5)` : requête → zembed-1 (`input_type="query"`) → cosinus → top-K publications.
Les chercheurs sont classés en agrégeant les similarités de leurs **3 meilleures publications** retrouvées (score = moyenne, places manquantes = 0),
et non par correspondance de mots avec le nom du laboratoire.

In [ ]:
engine = None
if store is not None and client is not None and articles is not None:
    from src.search.semantic_search import SemanticSearchEngine
    engine = SemanticSearchEngine(store, client, articles, researchers, links)
    QUERIES = ["deep learning for medical imaging", "natural language processing", "machine learning for cancer diagnosis",
               "renewable energy materials", "environmental pollution"]
    for q in QUERIES:
        display(Markdown(f"**« {q} »**"))
        display(pd.DataFrame([r.to_dict() for r in engine.search(q, settings["search"]["top_k"])])
                [["rank", "score", "titre", "chercheurs", "annee", "journal", "laboratoires", "citations"]])
else:
    print("Recherche non exécutée : il faut l'index (scripts 03-05) ET le cache d'embeddings zembed-1 (notebook Colab / script 04).")

## 15. Exemples de résultats
Recherche de **chercheurs** (« Quels chercheurs de la FSBM travaillent sur le NLP ? »).

In [ ]:
if engine is not None:
    for q in ["natural language processing", "artificial intelligence applied to medicine"]:
        display(Markdown(f"**« {q} »**"))
        display(pd.DataFrame([r.to_dict() for r in engine.search_researchers(
            q, 5, settings["search"]["researcher_pool"], settings["search"]["researcher_top_m"])])
                [["rank", "score", "nom_complet", "laboratoire", "n_matching_publications"]])
elif (REP / "demo_search_results.json").exists():
    print("Résultats enregistrés par scripts/06_demo_search.py :")
    for res in read_json(REP / "demo_search_results.json")[:3]:
        print("\n«", res["query"], "»")
        for r in res["publications"]:
            print("  ", r["rank"], f"{r['score']:.3f}", r["titre"])

## 16. Cartographie sémantique
Projection 2D (UMAP, `random_state=42`) + KMeans (k choisi par silhouette). Les clusters sont des **groupes sémantiques obtenus
mathématiquement**, pas des « vérités scientifiques » ni une taxonomie officielle.

In [ ]:
if vectors is not None and articles is not None and len(articles) >= 3:
    from src.analysis.semantic_map import build_semantic_map
    cfg = settings["semantic_map"]
    aligned = articles.set_index("article_id", drop=False).loc[manifest["article_id"]].reset_index(drop=True)
    res = build_semantic_map(aligned, vectors, links, researchers, FIG, REP, method=cfg["reduction"],
                             n_clusters=cfg["n_clusters"], k_range=tuple(cfg["k_range"]),
                             n_neighbors=cfg["umap_n_neighbors"], min_dist=cfg["umap_min_dist"],
                             random_state=settings["project"]["random_state"])
    print(res["info"])
    display(res["clusters"][["cluster", "n_publications", "top_terms", "titres_representatifs"]])
    display(Image(filename=res["paths"]["png_clusters"], width=800))
    print("Version interactive :", res["paths"]["html_clusters"])

## 17. Limites
* Google Scholar peut **modifier son HTML** ou limiter les requêtes ; en cas de blocage le pipeline s'arrête proprement et reprend plus tard (aucun contournement de CAPTCHA) ;
* les **citations évoluent** dans le temps : les valeurs correspondent à la date de collecte ;
* certaines métadonnées (abstract, DOI, date précise) peuvent être absentes ; l'aperçu Scholar est parfois tronqué (`abstract_status = truncated`) ;
* l'appariement de profils est probabiliste : les cas ambigus nécessitent une vérification humaine (`scholar_overrides.csv`) ;
* le plafond de publications par chercheur (`max_publications_per_researcher`) rend les classements **descriptifs du dataset collecté** ;
* les clusters dépendent de UMAP/KMeans et de l'échantillon : ce sont des regroupements sémantiques, pas des disciplines officielles ;
* l'ordre nom/prénom du PDF n'est pas déductible de façon fiable : les comparaisons de noms sont insensibles à l'ordre.

## 18. Conclusion
Le pipeline produit un dataset structuré et validé, un index vectoriel basé sur **zembed-1**, un moteur de recherche sémantique de
publications et de chercheurs, et une cartographie des travaux — avec une traçabilité complète des valeurs manquantes.